In [4]:
import re
import torch
import torch.nn as nn
import torch.utils.data as data
import torch.optim as optim
from navec import Navec
from tqdm import tqdm

In [5]:
path = 'navec_hudlit_v1_12B_500K_300d_100q.tar'
navec = Navec.load(path)

In [6]:
class WordsDataset(data.Dataset):
    def __init__(self, navec_emb, batch_size=8):
        self.batch_size = batch_size
        self.navec_emb = navec_emb
        
        with open(r"data\train_data_false.txt", encoding='utf-8') as d_f:
            phrase_false = d_f.readlines()
            self._clear_phrase(phrase_false)
            # print(phrase_false)
        
        with open(r"data\train_data_true.txt", encoding='utf-8') as d_t:
            phrase_true = d_t.readlines()
            self._clear_phrase(phrase_true)
            # print(phrase_true)
        
        self.all_data = [(x_n, 0) for x_n in phrase_false] + [(x_p, 1) for x_p in phrase_true]
        self.all_data.sort(key=lambda x: len(x[0]))
        self.dataset_len = len(self.all_data)
    
    
    def _clear_phrase(self, p_lst):
        for _i, _p in enumerate(p_lst):
            _p = _p.lower().replace('\ufeff', '').strip()
            _p = re.sub(r'[^А-яA-z- ]', '', _p)
            _words = _p.split()
            _words = [w for w in _words if w in self.navec_emb]
            p_lst[_i] = _words
    
    
    def __getitem__(self, item):
        item *= self.batch_size
        last_item = item + self.batch_size
        if last_item > self.dataset_len:
            last_item = self.dataset_len

        data = []
        target_list = []
        max_len_word = len(self.all_data[last_item - 1][0])
        
        for i in range(item, last_item):
            sentense = self.all_data[i][0]
            metka = self.all_data[i][1]
            
            emb_list = []
            
            difference = max_len_word - len(sentense)
            
            for word in sentense:
                emb_list.append(torch.from_numpy(self.navec_emb[word]))
            
            if difference > 0:
                for empty in range(difference):
                    emb_list.append(torch.zeros(300))
            
            data.append(torch.stack(emb_list))
            target_list.append(metka)

        data_batch = torch.stack(data)
        target_batch = torch.tensor(target_list, dtype=torch.float32)
        return data_batch, target_batch
    
    def __len__(self):
        last = 0 if self.dataset_len % self.batch_size == 0 else 1
        return self.dataset_len // self.batch_size + last

In [7]:
class WordsRNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.gru = nn.RNN(300, 16, batch_first=True, bidirectional=True) # GRU показывает более плохие результаты
        self.out = nn.Linear(32, 1)
    def forward(self, x):
        y, h = self.gru(x)
        hh = torch.cat((h[-2, :, :], h[-1, :, :]), dim=1)
        y = self.out(hh)
        return y

In [8]:
d_train = WordsDataset(navec_emb=navec)
train_data = data.DataLoader(d_train, batch_size=1, shuffle=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

epochs = 1000
model = WordsRNN()
model = model.to(device)
optimizer = optim.Adam(params=model.parameters(), lr=0.001, weight_decay=0.001)
loss_func = nn.BCEWithLogitsLoss()
best_loss = float('inf')
patience = 30
patience_counter = 0

model.train()
for _e in range(epochs):
    loss_mean = 0
    lm_count = 0

    train_tqdm = tqdm(train_data, leave=True)
    for x_train, y_train in train_tqdm:
        x_train, y_train = x_train.to(device), y_train.to(device)
        predict = model(x_train.squeeze(0)).squeeze()
        loss = loss_func(predict, y_train.squeeze())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        lm_count += 1
        loss_mean = 1/lm_count * loss.item() + (1 - 1/lm_count) * loss_mean
        train_tqdm.set_description(f"Epoch [{_e+1}/{epochs}], loss_mean={loss_mean:.3f}")
        
    if loss_mean < best_loss:
        best_loss = loss_mean
        patience_counter = 0
        torch.save(model.state_dict(), "model_rnn_bidir.tar")
    else:
        patience_counter += 1
    
    if patience_counter >= patience:
        print(f'Процесс прекращен на эпохе {_e + 1}')
        break

Epoch [148/1000], loss_mean=0.002: 100%|██████████| 22/22 [00:00<00:00, 146.55it/s]

Процесс прекращен на эпохе 148


In [9]:
best_weights = torch.load("model_rnn_bidir.tar", weights_only=True)
model.load_state_dict(best_weights)
model.eval()
phrase = "Ты - моя восходящая звезда"
phrase_lst = phrase.lower().split()
phrase_lst = [torch.tensor(navec[w]) for w in phrase_lst if w in navec]
data_batch_phrs = torch.stack(phrase_lst)
data_batch_phrs = data_batch_phrs.to(device)
predict = model(data_batch_phrs.unsqueeze(0)).squeeze(0)
p = torch.nn.functional.sigmoid(predict).item()
print(p)
print(phrase, ":", "положительное" if p > 0.5 else "отрицательное")

0.9952795505523682
Ты - моя восходящая звезда : положительное
